# 词向量  
把单词符号转化为一串固定长度的浮点数数组，这串数字就是这个单词的词向量。计算机无法直接理解文字，只能运算数字；词向量就是文字的数字化身，编码了单词的语义信息。   
good   → [ 0.23,  0.51, -0.14 ]  
great  → [ 0.21,  0.48, -0.11 ]  
bad    → [-0.27, -0.49,  0.16 ]  
movie  → [ 0.42,  0.08, -0.33 ]  

Word2vec 由谷歌于2013年发布，是一种神经网络实现，能够学习单词的分布式表示。此前已有其他深度或循环神经网络架构被提出用于学习词表表示，但主要问题是训练模型所需的时间过长。Word2vec 相较于其他模型学习速度更快。

Word2Vec 不需要标签来创建有意义的表示。这很有用，因为现实世界中大多数数据都没有标签。如果网络获得足够的训练数据（数百亿字），它会产生具有有趣特征的词向量。含义相似的词汇以簇的形式出现，且簇间距允许某些词语关系，如类比，可以用向量数学重现。著名的例子是，使用高度训练的词语向量，“king - man + woman = queen”。

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


## 准备训练模型  

In [2]:
import pandas as pd

train = pd.read_csv( "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip", header=0, 
 delimiter="\t", quoting=3 )
test = pd.read_csv( "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip", header=0,
                   delimiter="\t", quoting=3 )
unlabeled_train = pd.read_csv( "/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip", 
                              header=0, 
 delimiter="\t", quoting=3 )


print ("Read %d labeled train reviews, %d labeled test reviews, " \
 "and %d unlabeled reviews\n" % (train["review"].size,  
 test["review"].size, unlabeled_train["review"].size ))

Read 25000 labeled train reviews, 25000 labeled test reviews, and 50000 unlabeled reviews



## 清理数据

In [ ]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords

def review_to_wordlist(review,remove_stopwords=False):
    #1.去除HTML标签
    review_text= BeautifulSoup(review).get_text()
    #2.处理标点和数字
    letter_only = re.sub("[^a-zA-Z]"," ",review_text)
    #3.大写转小写，划分成单个单词
    lower_case = letter_only.lower()
    words = lower_case.split()
    #4.决定是否去除停止词
    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words = [w for w in words if not w in stops]
    return (words)


接下来，我们需要一个特定的输入格式。Word2Vec 期望有单句，每句都是单词列表。换句话说，输入格式是列表的列表。

In [ ]:
import nltk.data
nltk.download('punkt')

#加载 punkt 分句器
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

def review_to_sentences( review, tokenizer, remove_stopwords=False ):
    raw_sentences = tokenizer.tokenize(review.strip())
    sentences=[]
    for raw_sentence in raw_sentences:
        if len(raw_sentence)>0:
            sentences.append(review_to_wordlist(raw_sentence,remove_stopwords))
    return sentences

以上只是针对单条评论而言，现在应用这个函数来准备数据输入到 Word2Vec

In [22]:
sentences=[]
print("Parsing sentences from training set")
for review in train["review"]:
    sentences += review_to_sentences(review,tokenizer)  
    
print ("Parsing sentences from unlabeled set")
for review in unlabeled_train["review"]:
    sentences += review_to_sentences(review,tokenizer)

Parsing sentences from training set
Parsing sentences from unlabeled set


看看输出

In [25]:
print (len(sentences))
print (sentences[0])
print (sentences[1])

796172
['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the', 'odd', 'documentary', 'here', 'and', 'there', 'watched', 'the', 'wiz', 'and', 'watched', 'moonwalker', 'again']
['maybe', 'i', 'just', 'want', 'to', 'get', 'a', 'certain', 'insight', 'into', 'this', 'guy', 'who', 'i', 'thought', 'was', 'really', 'cool', 'in', 'the', 'eighties', 'just', 'to', 'maybe', 'make', 'up', 'my', 'mind', 'whether', 'he', 'is', 'guilty', 'or', 'innocent']


## 训练并保存模型

架构 (Architecture)：架构可选 skip‑gram（默认）或者连续词袋（CBOW）。实验发现 skip‑gram 速度会稍慢一点，但生成的效果更好。  
训练算法 (Training algorithm)：分层 Softmax（默认）或者负采样。本实验中，默认选项表现良好。
高频词下采样 (Downsampling of frequent words)：谷歌官方文档建议取值在 0.00001～0.001。本实验中，取更接近 0.001 的值，最终模型的准确率会有所提升。  
词向量维度 (Word vector dimensionality)：特征维度越大，训练耗时越长；维度更高通常（但并非一定）能得到效果更好的模型。合理取值在几十到几百之间；本文使用 300 维。  
上下文窗口大小 (Context /window size)：训练算法需要参考多少个上下文单词。对于分层 Softmax，窗口大小取 10 效果不错（在一定范围内越大越好）。  
工作线程数 (Worker threads)：并行进程数量。该参数取决于机器硬件，大多数设备设置为 4‑6 即可。  
最小词频 (Minimum word count)：用于把词表限制为有实际意义的词汇。在全部文档中出现次数低于该阈值的单词会被直接忽略。合理取值区间为 10‑100。    
本案例中，由于每部电影会出现 30 次，我们将最小词频设置为 40，避免给个别电影片名赋予过高权重。最终词表规模约 15000 个词。调高该参数也有助于缩短训练时间。

In [29]:
# 导入Python内置的logging模块并对其进行配置，以便生成格式友好的输出日志信息
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',\
    level=logging.INFO)

#为各个参数设置数值
num_features = 300    # 词向量维度               
min_word_count = 40   # 词的最小出现频次                  
num_workers = 12      # 并行运行线程数
context = 10          # 上下文窗口大小
downsampling = 1e-3   # 高频词下采样阈值

# 初始化和训练模型
from gensim.models import word2vec
print ("Training model...")
model = word2vec.Word2Vec(sentences, workers=num_workers, \
            vector_size=num_features, min_count = min_word_count, \
            window = context, sample = downsampling)

model.init_sims(replace=True)

model_name = "300features_40minwords_10context"
model.save(model_name)

2026-08-06 08:35:39,966 : INFO : collecting all words and their counts
2026-08-06 08:35:39,968 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-08-06 08:35:40,016 : INFO : PROGRESS: at sentence #10000, processed 225664 words, keeping 17775 word types
2026-08-06 08:35:40,065 : INFO : PROGRESS: at sentence #20000, processed 451738 words, keeping 24945 word types
2026-08-06 08:35:40,117 : INFO : PROGRESS: at sentence #30000, processed 670859 words, keeping 30027 word types
2026-08-06 08:35:40,168 : INFO : PROGRESS: at sentence #40000, processed 896841 words, keeping 34335 word types


Training model...


2026-08-06 08:35:40,219 : INFO : PROGRESS: at sentence #50000, processed 1116082 words, keeping 37751 word types
2026-08-06 08:35:40,271 : INFO : PROGRESS: at sentence #60000, processed 1337544 words, keeping 40711 word types
2026-08-06 08:35:40,322 : INFO : PROGRESS: at sentence #70000, processed 1560307 words, keeping 43311 word types
2026-08-06 08:35:40,374 : INFO : PROGRESS: at sentence #80000, processed 1779516 words, keeping 45707 word types
2026-08-06 08:35:40,425 : INFO : PROGRESS: at sentence #90000, processed 2003714 words, keeping 48121 word types
2026-08-06 08:35:40,476 : INFO : PROGRESS: at sentence #100000, processed 2225465 words, keeping 50190 word types
2026-08-06 08:35:40,526 : INFO : PROGRESS: at sentence #110000, processed 2444323 words, keeping 52058 word types
2026-08-06 08:35:40,574 : INFO : PROGRESS: at sentence #120000, processed 2666488 words, keeping 54098 word types
2026-08-06 08:35:40,625 : INFO : PROGRESS: at sentence #130000, processed 2892315 words, keep

## 探索模型结果

“doesnt_match”函数将尝试推断集合中哪个词与其他词最不相似：

In [33]:
print(model.wv.doesnt_match("dog woman cat tiger".split()))
print(model.wv.doesnt_match("beijing shanghai phone berlin".split()))

woman
phone


我们也可以使用“most_similar”函数来洞察模型的词群：

In [35]:
print(model.wv.most_similar("book"))
print(model.wv.most_similar("happy"))

[('books', 0.7637225389480591), ('novel', 0.7540881633758545), ('novels', 0.5642451643943787), ('manga', 0.5311359167098999), ('poem', 0.46819597482681274), ('comics', 0.46179717779159546), ('review', 0.457421213388443), ('adaptation', 0.45360231399536133), ('description', 0.4490889310836792), ('biography', 0.42896080017089844)]
[('satisfied', 0.47666335105895996), ('afraid', 0.456387460231781), ('lucky', 0.45317989587783813), ('pleased', 0.4238404333591461), ('sad', 0.4056549370288849), ('proud', 0.4044855535030365), ('upset', 0.3996661305427551), ('glad', 0.3994075059890747), ('ready', 0.3982633650302887), ('thrilled', 0.39206674695014954)]
